# 01 — Why Memory Matters

**No LLM. No API key. Just Python.**

This notebook explains the core problem that memory solves, then builds a simple
in-memory store from scratch so you can see exactly what "memory" means for an AI agent.

### What you will learn
| Step | Concept |
|------|---------|
| 1 | Stateless chatbot — forgets everything between messages |
| 2 | What memory actually is — just a Python dict/list |
| 3 | Remember facts about the user |
| 4 | Store a full conversation history |
| 5 | Use that history to give smarter replies |

> No external libraries needed beyond Python's standard library.

## 1. The Problem — A Chatbot With No Memory

In [ ]:
# This chatbot has a fixed lookup table.
# It cannot remember anything you told it earlier.

REPLIES = {
    "hello":            "Hi there! How can I help?",
    "what is my name":  "I don't know your name.",   # ← the problem!
    "how old am i":     "I have no idea.",            # ← same problem
    "bye":              "Goodbye!",
}

def forgetful_bot(user_input):
    reply = REPLIES.get(user_input.lower(), "Sorry, I don't understand.")
    print(f"  You : {user_input}")
    print(f"  Bot : {reply}")
    print()

# Simulate a short conversation
forgetful_bot("hello")
forgetful_bot("My name is Priya")          # tells the bot something
forgetful_bot("what is my name")           # bot still can't answer!
forgetful_bot("I am 25 years old")
forgetful_bot("how old am i")              # still no answer!

## 2. What IS Memory?

Memory is just **data stored somewhere** that the agent can read later.

The simplest possible memory is a Python dictionary.

In [ ]:
# The simplest memory: a plain Python dictionary
memory = {}

# Store something
memory["user_name"] = "Priya"
memory["user_age"]  = 25
memory["user_city"] = "Mumbai"

# Read it back
print("What I know about the user:")
for key, value in memory.items():
    print(f"  {key}: {value}")

# Check if something is remembered
if "user_name" in memory:
    print(f"\nI remember your name: {memory['user_name']}")
else:
    print("\nI don't know your name yet.")

## 3. Build a Proper Memory Store

In [ ]:
class SimpleMemory:
    """A basic key-value memory store for an AI agent."""

    def __init__(self):
        self.facts   = {}          # user facts: name, age, city, etc.
        self.history = []          # list of all messages

    # ── Storing facts ──────────────────────────────────────
    def remember(self, key, value):
        self.facts[key] = value
        print(f"  [Memory] Saved: {key} = {value}")

    def recall(self, key):
        return self.facts.get(key, None)

    # ── Conversation history ───────────────────────────────
    def add_message(self, role, text):
        self.history.append({"role": role, "text": text})

    def get_history(self):
        return self.history

    def show_history(self):
        print("\n--- Conversation History ---")
        for msg in self.history:
            label = "You" if msg["role"] == "user" else "Bot"
            print(f"  {label}: {msg['text']}")
        print("----------------------------")

    def clear(self):
        self.facts   = {}
        self.history = []
        print("  [Memory] Cleared all memory.")


# Create a memory instance
mem = SimpleMemory()
print("Memory store created!")
print(f"Facts stored: {mem.facts}")
print(f"History stored: {mem.history}")

## 4. A Chatbot That Uses Memory

In [ ]:
def smart_bot(user_input, memory):
    """
    A simple rule-based bot that reads from and writes to memory.
    No LLM needed — just Python logic.
    """
    text = user_input.strip()
    memory.add_message("user", text)

    reply = ""

    # ── Learn user's name ──────────────────────────────────
    if "my name is" in text.lower():
        name = text.lower().split("my name is")[-1].strip().split()[0].title()
        memory.remember("name", name)
        reply = f"Nice to meet you, {name}! I'll remember that."

    # ── Learn user's age ───────────────────────────────────
    elif "i am" in text.lower() and "years old" in text.lower():
        words = text.lower().split()
        for i, w in enumerate(words):
            if w == "am" and i + 1 < len(words) and words[i+1].isdigit():
                memory.remember("age", int(words[i+1]))
                reply = f"Got it! I'll remember you're {words[i+1]} years old."
                break
        if not reply:
            reply = "I heard you mention your age but couldn't catch the number."

    # ── Answer questions using memory ──────────────────────
    elif "what is my name" in text.lower() or "what's my name" in text.lower():
        name = memory.recall("name")
        reply = f"Your name is {name}!" if name else "You haven't told me your name yet."

    elif "how old am i" in text.lower():
        age = memory.recall("age")
        reply = f"You are {age} years old." if age else "You haven't told me your age yet."

    elif "what do you know about me" in text.lower():
        if memory.facts:
            known = ", ".join(f"{k}: {v}" for k, v in memory.facts.items())
            reply = f"Here's what I know about you — {known}"
        else:
            reply = "You haven't told me anything about yourself yet."

    elif text.lower() in ("hello", "hi", "hey"):
        name = memory.recall("name")
        reply = f"Hello, {name}!" if name else "Hello! What's your name?"

    else:
        reply = "Interesting! Tell me more about yourself."

    memory.add_message("bot", reply)
    print(f"  You : {text}")
    print(f"  Bot : {reply}")
    print()
    return reply

## 5. Run the Memory-Powered Chatbot

In [ ]:
mem = SimpleMemory()

print("=== Conversation with Memory ===\n")
smart_bot("Hi!", mem)
smart_bot("My name is Arjun", mem)
smart_bot("I am 22 years old", mem)
smart_bot("What is my name?", mem)
smart_bot("How old am I?", mem)
smart_bot("What do you know about me?", mem)

## 6. Inspect What's In Memory

In [ ]:
print("Facts stored in memory:")
for key, value in mem.facts.items():
    print(f"  {key:12s} → {value}")

mem.show_history()

## 7. What Happens When Memory is Lost?

This shows why **persistent memory** (saving to disk) matters.
Without it, every time you restart Python — all memory is gone.

In [ ]:
# What a restart looks like — all memory lost
mem_after_restart = SimpleMemory()   # brand new, empty

print("After restart — trying to recall name:")
name = mem_after_restart.recall("name")
print(f"  Result: '{name}'")       # None — gone!

print("\nThis is why we need to SAVE memory to a file.")
print("We solve this in Notebook 05 — Persistent Memory.")

## 8. Key Takeaways

| Concept | What it means |
|---------|--------------|
| Stateless | Every message is processed independently — no context |
| Memory | Data stored (dict/list) that the agent can read later |
| Facts memory | Key-value store of things the user told us |
| History | List of every message exchanged, in order |
| Why it matters | Without memory, agents can't maintain a conversation |

**Next notebook →** use a real Groq LLM with a conversation history buffer.